In [28]:
!pip install ddgs requests beautifulsoup4 lxml pandas -q

In [8]:
!pip uninstall mistralai -y
!pip install mistralai -q

Found existing installation: mistralai 2.2.0
Uninstalling mistralai-2.2.0:
  Successfully uninstalled mistralai-2.2.0


In [6]:
import os
from getpass import getpass

os.environ["MISTRAL_API_KEY"] = getpass("🔑 Mistral API key: ")

🔑 Mistral API key: ··········


In [29]:
# =========================================
# IMPORTS
# =========================================
from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re

API_KEY = os.environ["MISTRAL_API_KEY"]

# =========================================
# 🔵 LLM
# =========================================
def llm(prompt):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "mistral-small",
        "messages": [
            {"role": "system", "content": "You extract structured data from messy web text."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    r = requests.post(url, json=data, headers=headers)

    if r.status_code != 200:
        return ""

    return r.json()["choices"][0]["message"]["content"]


# =========================================
# 🔎 SEARCH (MULTI QUERY)
# =========================================
def search(country):
    queries = [
        f"{country} central bank board members",
        f"{country} central bank governor deputy governor",
        f"{country} central bank council members",
        f"{country} banco central consejo miembros",
    ]

    links = []

    with DDGS() as ddgs:
        for q in queries:
            results = list(ddgs.text(q, max_results=5))

            for r in results:
                url = r["href"].lower()

                if any(x in url for x in ["wikipedia","news","reuters","pdf"]):
                    continue

                links.append(r["href"])

    return list(set(links))[:12]


# =========================================
# 🌐 GET TEXT
# =========================================
def get_text(url):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(r.text, "lxml")

        texts = [
            t.get_text(strip=True)
            for t in soup.find_all(["h1","h2","h3","p","li","span","div"])
        ]

        return "\n".join(texts[:1200])

    except:
        return ""


# =========================================
# 🔪 CHUNKING
# =========================================
def chunk_text(text, size=4000):
    return [text[i:i+size] for i in range(0, len(text), size)]


# =========================================
# 🧠 EXTRACT WITH MULTIPLE CALLS
# =========================================
def extract_all(text, country):

    chunks = chunk_text(text)

    outputs = []

    for chunk in chunks:
        prompt = f"""
        Extract ALL central bank board members of {country}.

        Important:
        - Do NOT summarize
        - Extract EVERY name
        - Include governor, deputy governor, directors

        Return format:
        Name - Role

        TEXT:
        {chunk}
        """

        out = llm(prompt)
        outputs.append(out)

    return "\n".join(outputs)


# =========================================
# 🧼 PARSE
# =========================================
def parse(text, country):
    rows = []

    for line in text.split("\n"):
        if "-" in line:
            parts = line.split("-", 1)

            name = parts[0].strip()
            role = parts[1].strip()

            if len(name) > 3:
                rows.append({
                    "country": country,
                    "name": name,
                    "role": role
                })

    return rows


# =========================================
# 🧹 DEDUP
# =========================================
def deduplicate(rows):
    seen = set()
    clean = []

    for r in rows:
        key = (r["country"], r["name"])

        if key not in seen:
            seen.add(key)
            clean.append(r)

    return clean


# =========================================
# 🧠 AGENT
# =========================================
def agent(country):

    links = search(country)

    combined = ""

    for link in links[:8]:   # más páginas
        combined += get_text(link) + "\n\n"

    extracted = extract_all(combined, country)

    parsed = parse(extracted, country)

    return deduplicate(parsed)


# =========================================
# 🌍 BUILD DATASET
# =========================================
def build_dataset(countries):

    all_rows = []

    for c in countries:
        print(f"🔍 {c}...")

        try:
            rows = agent(c)
            all_rows.extend(rows)

        except Exception as e:
            print("❌ Error:", e)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["country","name"])

    return df


# =========================================
# 🚀 RUN
# =========================================
countries = [
    "Chile",
    "Argentina",
    "Brazil",
    "United States",
    "European Central Bank"
]

df = build_dataset(countries)

print(df)

🔍 Chile...
🔍 Argentina...
🔍 Brazil...
🔍 United States...
🔍 European Central Bank...
                   country                                               name  \
0                    Chile                           **Luis Felipe Céspedes**   
1                    Chile                                **Joaquín Vial Ruiz   
2                    Chile                             **Luis Felipe Céspedes   
3                    Chile  It seems the provided text does not contain in...   
4                    Chile  *Note: The text mentions that Luis Felipe Césp...   
..                     ...                                                ...   
315  European Central Bank                                  **Robert Ophèle**   
316  European Central Bank                                   **Daniele Nouy**   
317  European Central Bank                              **Christine Lagarde**   
318  European Central Bank  It seems the provided text does not contain an...   
319  European Central Ban

In [ ]:
df.to_csv("central_bank_boards_full.csv", index=False)

In [31]:
df

,country,name,role
0,Chile,**Luis Felipe Céspedes**,"Board Member (confirmed on November 30, 2021, ..."
1,Chile,**Joaquín Vial Ruiz,Tagle** - Deputy Governor (term ending)
2,Chile,**Luis Felipe Céspedes,Board Member**
3,Chile,It seems the provided text does not contain in...,related news and events from different countri...
4,Chile,*Note: The text mentions that Luis Felipe Césp...,"Tagle, who is the current deputy governor. How..."
...,...,...,...
315,European Central Bank,**Robert Ophèle**,French central banker (mentioned as a past con...
316,European Central Bank,**Daniele Nouy**,Former Chair of the ECB’s Supervisory Board (p...
317,European Central Bank,**Christine Lagarde**,President of the European Central Bank
318,European Central Bank,It seems the provided text does not contain an...,European Council on Foreign Relations) and doe...


In [30]:
#uktra paper

In [ ]:
# =========================================
# IMPORTS
# =========================================
from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re

API_KEY = os.environ["MISTRAL_API_KEY"]


# =========================================
# 🔵 LLM (SOLO EXTRACCIÓN FINAL)
# =========================================
def llm_extract(text, country):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""
    Extract ALL central bank board members of {country}.

    STRICT:
    - No explanations
    - No summaries
    - Only names and roles

    Format:
    Name - Role

    TEXT:
    {text}
    """

    data = {
        "model": "mistral-small",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }

    r = requests.post(url, json=data, headers=headers)

    if r.status_code != 200:
        return ""

    return r.json()["choices"][0]["message"]["content"]


# =========================================
# 🔎 DETECT OFFICIAL SITE
# =========================================
def find_official_site(country):

    query = f"{country} central bank official website"

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))

    for r in results:
        url = r["href"].lower()

        if any(x in url for x in ["centralbank", "bcentral", ".gov"]):
            return r["href"]

    return results[0]["href"]


# =========================================
# 🔎 FIND BOARD PAGE
# =========================================
def find_board_page(base_url):

    keywords = ["board", "council", "governance", "directors", "consejo"]

    try:
        r = requests.get(base_url, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "lxml")

        for a in soup.find_all("a", href=True):
            href = a["href"].lower()

            if any(k in href for k in keywords):
                return href if href.startswith("http") else base_url + href

    except:
        pass

    return base_url


# =========================================
# 🌐 GET TEXT (LIMPIO)
# =========================================
def get_clean_text(url):

    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(r.text, "lxml")

        texts = []

        for tag in soup.find_all(["h1","h2","h3","li","p","span"]):
            t = tag.get_text(strip=True)

            if len(t) > 3:
                texts.append(t)

        return "\n".join(texts[:1500])

    except:
        return ""


# =========================================
# 🔁 FALLBACK SEARCH
# =========================================
def fallback_search(country):

    query = f"{country} central bank board members"

    links = []

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=8))

        for r in results:
            url = r["href"].lower()

            if "wikipedia" in url:
                continue

            links.append(r["href"])

    return links[:5]


# =========================================
# 🧼 PARSE
# =========================================
def parse(text, country):

    rows = []

    for line in text.split("\n"):
        if "-" in line:

            name, role = line.split("-", 1)

            rows.append({
                "country": country,
                "name": name.strip(),
                "role": role.strip()
            })

    return rows


# =========================================
# 🧹 DEDUP
# =========================================
def dedup(rows):

    seen = set()
    clean = []

    for r in rows:
        key = (r["country"], r["name"])

        if key not in seen:
            seen.add(key)
            clean.append(r)

    return clean


# =========================================
# 🧠 ULTRA AGENT
# =========================================
def ultra_agent(country):

    # 1. sitio oficial
    base = find_official_site(country)

    # 2. página board
    board_page = find_board_page(base)

    # 3. texto principal
    text = get_clean_text(board_page)

    # 4. fallback si falla
    if len(text) < 200:
        links = fallback_search(country)

        for link in links:
            text += get_clean_text(link) + "\n\n"

    # 5. extracción LLM
    extracted = llm_extract(text, country)

    # 6. parse
    parsed = parse(extracted, country)

    return dedup(parsed)


# =========================================
# 🌍 BUILD DATASET
# =========================================
def build_dataset(countries):

    all_rows = []

    for c in countries:
        print(f"🔍 {c}")

        try:
            rows = ultra_agent(c)
            all_rows.extend(rows)

        except Exception as e:
            print("❌", e)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["country","name"])

    return df


# =========================================
# 🚀 RUN
# =========================================
countries = [
    "Chile",
    "Argentina",
    "Brazil",
    "United States",
    "European Central Bank"
]

df = build_dataset(countries)

print(df)